# Notebook 3 of 3 - Traces and Evaluation
Owner: Quang Tran - PM

---

### PaySprint AI Investment Agent Workflow

| Step | Notebook | Owner | Output |
|------|----------|-------|--------|
| 1 | 'data_pipeline' | Reema Eid (Data Engineer) | Validated + ranked stock universe â†’ 'data/screener_override.json', 'data/master_stock_data.csv' |
| 2 | 'agent_definition' | Hyunju Yu (AI Engineer) | ReAct agent traces â†’ 'data/traces/trace_*.json' |
| **3** | **'traces_evaluation' (this notebook)** | **Quang Tran (PM)** | **LLM judge scores, backtesting, consistency analysis â†’ 'data/evaluation_results.csv'** |

---

### What this notebook does

This notebook picks up where Notebooks 1 and 2 left off:
- Loads the pipeline-validated stock universe from Notebook 1 for context
- Runs the agent 5 times with different investor scenarios (using traces from Notebook 2 as a baseline)
- Evaluates each report with an LLM judge on 4 dimensions (reasoning, risk alignment, clarity, plan accuracy)
- Backtests the conservative portfolio against SPY over 63 trading days
- Measures consistency across 3 repeated runs of the same profile

In [1]:
# === Setup: load BOTH providers ============================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os, json, threading
sys.path.insert(0, '.')

from openai import OpenAI
import paysprint_agent as core
from paysprint_agent import (
    run_agent, test_rejection, llm_judge,
    save_trace, load_trace, print_trace_summary,
    estimate_cost, backtest, consistency_test,
)
import pandas as pd

GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'
GEMINI_KEY = os.getenv('GEMINI_API_KEY', '')
OPENAI_KEY = os.getenv('OPENAI_API_KEY', '')

print('API keys detected:')
print(f'  GEMINI_API_KEY : {"FOUND" if GEMINI_KEY else "NOT FOUND"}')
print(f'  OPENAI_API_KEY : {"FOUND" if OPENAI_KEY else "NOT FOUND"}')

if not GEMINI_KEY and not OPENAI_KEY:
    raise EnvironmentError('No API keys found. Set GEMINI_API_KEY and/or OPENAI_API_KEY.')

# Thread-local storage so parallel threads can each use a different provider
# without interfering with each other.
_provider_local = threading.local()

def _dual_client(model=None):
    """Route to Gemini or OpenAI based on thread-local provider override."""
    provider = getattr(_provider_local, 'provider', 'gemini' if GEMINI_KEY else 'openai')
    if provider == 'gemini':
        if not GEMINI_KEY:
            raise EnvironmentError('GEMINI_API_KEY not set.')
        return OpenAI(api_key=GEMINI_KEY, base_url=GEMINI_BASE_URL)
    if not OPENAI_KEY:
        raise EnvironmentError('OPENAI_API_KEY not set.')
    return OpenAI(api_key=OPENAI_KEY)

core._get_client = _dual_client
print('\nDual-provider client ready.')

API keys detected:
  GEMINI_API_KEY : FOUND
  OPENAI_API_KEY : FOUND

Dual-provider client ready.


In [2]:
# === Model selection =========================================================
if GEMINI_KEY and OPENAI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')   # Gemini â€” main traces
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')        # OpenAI â€” comparison
elif GEMINI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gemini-2.5-flash-lite')
else:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gpt-4o')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')

core.MODEL_REASONING = MODEL_1
core.MODEL_SUMMARY   = MODEL_2

# Set main-thread default: Traces 1-4 all use MODEL_1
PROVIDER_1 = 'gemini' if MODEL_1.startswith('gemini') else 'openai'
PROVIDER_2 = 'gemini' if MODEL_2.startswith('gemini') else 'openai'
_provider_local.provider = PROVIDER_1

print(f'Model 1 â†’ {PROVIDER_1.upper()} : {MODEL_1}')
print(f'Model 2 â†’ {PROVIDER_2.upper()} : {MODEL_2}')
print(f'Main thread provider set to: {PROVIDER_1}')
print()
print('Pricing reference (per 1M tokens):')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')

Model 1 â†’ GEMINI : gemini-2.5-flash
Model 2 â†’ OPENAI : gpt-4o-mini
Main thread provider set to: gemini

Pricing reference (per 1M tokens):
  gpt-4o: $2.5/1M input, $10.0/1M output
  gpt-4o-mini: $0.15/1M input, $0.6/1M output
  gemini-2.5-flash: $0.3/1M input, $2.5/1M output
  gemini-2.5-flash-lite: $0.1/1M input, $0.4/1M output


---
## Step 0 â€” Pipeline Context from Notebook 1

Before running agent traces, we load the scoring output from Notebook 1 (data_pipeline)
to see what the data pipeline ranked across all candidate stocks.

This gives a reference point for the trace analysis: we can check whether the agent
consistently picks the stocks the pipeline scored highest, or whether the LLM reasoning
diverges from the quantitative signal.

In [3]:
# === Load pipeline data from Notebook 1 for context =========================
# master_stock_data.csv contains all candidate tickers scored by trend, sentiment,
# and fundamentals. We load it here to show what the data pipeline found, and
# later compare it against what the agent actually picked in each trace.
import os, pandas as pd

pipeline_path = 'data/master_stock_data.csv'
if os.path.exists(pipeline_path):
    pipeline_df = pd.read_csv(pipeline_path)
    print(f'Pipeline data loaded: {len(pipeline_df)} stocks scored by Notebook 1 (data_pipeline)')

    # Show the top-ranked stocks with key signals
    show_cols = [c for c in ['ticker', 'trend_score', 'avg_sentiment', 'slope_per_day',
                              'revenue_growth', 'risk_label', 'suggested_investment_amount',
                              'used_example_data'] if c in pipeline_df.columns]
    if show_cols and 'trend_score' in pipeline_df.columns:
        print('\nTop 10 pipeline-ranked stocks (by trend score):')
        print(pipeline_df[show_cols].sort_values('trend_score', ascending=False).head(10).to_string(index=False))
    print()
    print('These are the candidates the agent will research across the 5 traces below.')
else:
    pipeline_df = None
    print('No pipeline data found â€” run Notebook 1 (data_pipeline) first.')
    print('Continuing without pipeline context.')

Pipeline data loaded: 12 stocks scored by Notebook 1 (data_pipeline)

Top 10 pipeline-ranked stocks (by trend score):
ticker  trend_score  avg_sentiment  slope_per_day  revenue_growth risk_label  suggested_investment_amount  used_example_data
  NVDA        0.822         -0.013         0.3002           0.852      Lower                       1020.0              False
  AVGO        0.773          0.031         0.9241           0.479     Medium                        700.0              False
  MRVL        0.754          0.214         1.6186           0.276       High                        370.0              False
  INTC        0.741          0.169         0.7666           0.072       High                        370.0              False
  ORCL        0.741          0.105         0.2126           0.206       High                        370.0              False
    MU        0.728          0.027         5.3422           1.963       High                        370.0              False
  SNDK 

---
## The 5 Traces

| Trace | Scenario | Model | Purpose |
|-------|----------|-------|---------|
| 1 | Conservative investor, $5,000, 12 months | MODEL_1 | Normal run, low-risk strategy |
| 2 | Aggressive investor, $10,000, 6 months | MODEL_1 | High-risk strategy |
| 3 | Moderate investor, $3,000, 24 months (holds AAPL) | MODEL_1 | Balanced, existing holdings |
| 4 | Off-topic rejection tests | MODEL_1 | Graceful rejection |
| 5 | Same profile as Trace 1, MODEL_1 vs MODEL_2 | Both | LLM comparison |

---
## Trace 1 - Conservative Investor ($5,000, 12 months)

In [4]:
PROFILE_1 = {
    'name':              'Conservative Carol',
    'budget':            5000.0,
    'aggressiveness':    'conservative',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': [],
}

print('=== TRACE 1: Conservative $5k / 12mo ===\n')
trace1 = run_agent(PROFILE_1, model=MODEL_1, max_turns=20, verbose=True)
trace1.update({'trace_id': 1, 'label': f'Conservative $5k 12mo [{MODEL_1}]', 'profile': PROFILE_1})
save_trace(trace1, trace_id=1)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace1.get('report', ''))
print_trace_summary(trace1)

=== TRACE 1: Conservative $5k / 12mo ===

  [Turn 1]  screen_stocks({"aggressiveness": "conservative"}...)
             -> {"candidates": ["JNJ", "PG", "KO", "VZ", "WMT", "MCD", "MMM", "ABT"], "strategy": "conservative"}...
  [Turn 2]  get_technical_indicators({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "last_price": 234.2, "slope_per_day": 0.1227, "forecast_3m": 241.93, "forecast_12m...
  [Turn 2]  get_news_sentiment({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "avg_sentiment": 0.243, "article_count": 30, "top_headlines": ["Johnson & Johnson ...
  [Turn 2]  get_fundamentals({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "pe_ratio": 27.13789, "eps": 8.63, "revenue_growth": 0.099, "debt_to_equity": 67.7...
  [Turn 3]  get_technical_indicators({"ticker": "PG"}...)
             -> {"ticker": "PG", "last_price": 150.56, "slope_per_day": -0.0324, "forecast_3m": 148.52, "forecast_12...
  [Turn 3]  get_news_sentiment({"ticker": "PG"}...)
             -> {"ticke

---
## Trace 2 - Aggressive Investor ($10,000, 6 months)

In [5]:
PROFILE_2 = {
    'name':              'Aggressive Alex',
    'budget':            10000.0,
    'aggressiveness':    'aggressive',
    'horizon_months':    6,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

print('=== TRACE 2: Aggressive $10k / 6mo ===\n')
trace2 = run_agent(PROFILE_2, model=MODEL_1, max_turns=20, verbose=True)
trace2.update({'trace_id': 2, 'label': f'Aggressive $10k 6mo [{MODEL_1}]', 'profile': PROFILE_2})
save_trace(trace2, trace_id=2)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace2.get('report', ''))
print_trace_summary(trace2)

=== TRACE 2: Aggressive $10k / 6mo ===

  [Turn 1]  screen_stocks({"aggressiveness": "aggressive", "sectors": ["Technology"]}...)
             -> {"candidates": ["NVDA", "META", "TSLA", "AMD", "PLTR", "CRWD", "SNOW", "SMCI"], "strategy": "aggress...
  [Turn 2]  get_technical_indicators({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "last_price": 204.65, "slope_per_day": 0.3002, "forecast_3m": 223.56, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "avg_sentiment": 0.0, "article_count": 2, "top_headlines": ["What's Going On With...
  [Turn 2]  get_fundamentals({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "pe_ratio": 31.339968, "eps": 6.53, "revenue_growth": 0.852, "debt_to_equity": 6....
  [Turn 3]  get_technical_indicators({"ticker": "META"}...)
             -> {"ticker": "META", "last_price": 567.58, "slope_per_day": -0.5375, "forecast_3m": 533.71, "forecast_...
  [Turn 3]  get_news_sentiment({"ticker": "M

---
## Trace 3 - Moderate Investor ($3,000, 24 months, already holds AAPL)

In [6]:
PROFILE_3 = {
    'name':              'Moderate Mike',
    'budget':            3000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    24,
    'current_holdings':  {'AAPL': 3},
    'preferred_sectors': [],
}

print('=== TRACE 3: Moderate $3k / 24mo (holds AAPL) ===\n')
trace3 = run_agent(PROFILE_3, model=MODEL_1, max_turns=20, verbose=True)
trace3.update({'trace_id': 3, 'label': f'Moderate $3k 24mo [{MODEL_1}]', 'profile': PROFILE_3})
save_trace(trace3, trace_id=3)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace3.get('report', ''))
print_trace_summary(trace3)

=== TRACE 3: Moderate $3k / 24mo (holds AAPL) ===

  [Turn 1]  screen_stocks({"aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA", "AMZN", "JPM", "HD"], "strategy": "moderate"}...
  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 295.95, "slope_per_day": 0.37, "forecast_3m": 319.26, "forecast_12m...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.257, "article_count": 30, "top_headlines": ["Wall Street\u2019...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.8293, "eps": 8.26, "revenue_growth": 0.166, "debt_to_equity": 79.5...
  [Turn 2]  get_technical_indicators({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "last_price": 378.91, "slope_per_day": -0.4259, "forecast_3m": 352.08, "forecast_...
  [Turn 2]  get_news_sentiment({"ticker": "MSFT"}...)
           

---
## Trace 4 - Graceful Rejection (2 examples)

The agent must refuse irrelevant questions without calling any tools.

In [7]:
rejection_inputs = [
    'What is the capital of France?',
    'Can you write me a Python script to sort a list?',
]

print('=== TRACE 4: Graceful Rejection Tests ===\n')
rejection_results = []
for i, msg in enumerate(rejection_inputs, 1):
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called!)'
    print(f'Rejection {i}: {status}')
    print(f'  Input:    "{msg}"')
    print(f'  Response: {r["response"][:300]}')
    print()

save_trace({'rejection_tests': rejection_results, 'trace_id': '4_rejection'}, trace_id='4_rejection')
print('Saved Trace 4')

=== TRACE 4: Graceful Rejection Tests ===

Rejection 1: PASS (no tools called)
  Input:    "What is the capital of France?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.

Rejection 2: PASS (no tools called)
  Input:    "Can you write me a Python script to sort a list?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.

[trace] Saved -> data\traces\trace_4_rejection.json
Saved Trace 4


---
## Trace 5 - LLM Comparison (same profile, two models)

The same investor profile runs through both models so we can compare quality and cost.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

COMPARE_PROFILE = PROFILE_1.copy()
COMPARE_PROFILE['name'] = 'Compare Investor'

print(f'Running both models IN PARALLEL...')
print(f'  Model 1: {MODEL_1}  ({PROVIDER_1.upper()})')
print(f'  Model 2: {MODEL_2}  ({PROVIDER_2.upper()})')
print('-' * 60)

def run_model(label, model, provider):
    # Each thread sets its own provider â€” thread-local so threads don't interfere
    _provider_local.provider = provider
    print(f'[{label}] started on {provider.upper()}...')
    result = run_agent(COMPARE_PROFILE, model=model, max_turns=20, verbose=False)
    tokens = result.get('usage', {}).get('total_tokens', 0)
    print(f'[{label}] done  ({result.get("turns")} turns, {tokens:,} tokens)')
    return label, result

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {
        pool.submit(run_model, 'Model 1', MODEL_1, PROVIDER_1): 'Model 1',
        pool.submit(run_model, 'Model 2', MODEL_2, PROVIDER_2): 'Model 2',
    }
    results_map = {}
    for future in as_completed(futures):
        label, result = future.result()
        results_map[label] = result

trace5a = results_map['Model 1']
trace5b = results_map['Model 2']
trace5a.update({'trace_id': '5a', 'label': f'Compare [{MODEL_1}]', 'profile': COMPARE_PROFILE})
trace5b.update({'trace_id': '5b', 'label': f'Compare [{MODEL_2}]', 'profile': COMPARE_PROFILE})
save_trace(trace5a, trace_id='5a')
save_trace(trace5b, trace_id='5b')
print('\nBoth traces saved.')

Running both models IN PARALLEL...
  Model 1: gemini-2.5-flash  (GEMINI)
  Model 2: gpt-4o-mini  (OPENAI)
------------------------------------------------------------
[Model 1] started on GEMINI...
[Model 2] started on OPENAI...


Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\andre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gnews\utils\utils.py", line 38, in _resolve_with_playwright
    with sync_playwright() as p:
  File "C:\Users\andre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\playwright\sync_api\_context_manager.py", line 77, in __enter__
    dispatcher_fiber.switch()
  File "C:\Users\andre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\playwright\sync_api\_context_manager.py", line 56, in greenlet_main
    self._loop.run_until_complete(self._connection.run_as_sync())
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib

In [9]:
# === Side-by-side comparison =================================================
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

W = 50  # column width

def col(val, width=W):
    return str(val)[:width].ljust(width)

# Pre-format values to avoid f-string nesting with backslashes
tot_a   = f'{cost5a["input_tokens"] + cost5a["output_tokens"]:,}'
tot_b   = f'{cost5b["input_tokens"] + cost5b["output_tokens"]:,}'
inp_a   = f'{cost5a["input_tokens"]:,}'
inp_b   = f'{cost5b["input_tokens"]:,}'
out_a   = f'{cost5a["output_tokens"]:,}'
out_b   = f'{cost5b["output_tokens"]:,}'
cost_a  = f'${cost5a["cost_usd"]:.6f}'
cost_b  = f'${cost5b["cost_usd"]:.6f}'
label_a = f'Model 1: {MODEL_1} ({PROVIDER_1.upper()})'
label_b = f'Model 2: {MODEL_2} ({PROVIDER_2.upper()})'

print('=' * 120)
print('  SIDE-BY-SIDE COMPARISON')
print(f'  Profile: {COMPARE_PROFILE["name"]} | ${COMPARE_PROFILE["budget"]:,.0f} | {COMPARE_PROFILE["aggressiveness"]} | {COMPARE_PROFILE["horizon_months"]}mo')
print('=' * 120)

# --- Stats table ---
print(f'\n  {"Metric":<22} {col(label_a)} {col(label_b)}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
print(f'  {"Turns":<22} {col(trace5a.get("turns"))} {col(trace5b.get("turns"))}')
print(f'  {"Total tokens":<22} {col(tot_a)} {col(tot_b)}')
print(f'  {"Input tokens":<22} {col(inp_a)} {col(inp_b)}')
print(f'  {"Output tokens":<22} {col(out_a)} {col(out_b)}')
print(f'  {"Est. cost (USD)":<22} {col(cost_a)} {col(cost_b)}')

# --- Tools called ---
tools5a = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5a.get('tool_calls', [])]
tools5b = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5b.get('tool_calls', [])]
print(f'\n  {"TOOLS CALLED":<22} {col("Model 1")} {col("Model 2")}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
for i in range(max(len(tools5a), len(tools5b))):
    a = tools5a[i] if i < len(tools5a) else ''
    b = tools5b[i] if i < len(tools5b) else ''
    print(f'  {"":<22} {col(a)} {col(b)}')

# --- Reports ---
print(f'\n{"=" * 120}')
print('  FINAL REPORTS')
print('=' * 120)

print(f'\n  {label_a}')
print(f'  {"-" * 60}')
for line in trace5a.get('report', '').strip().split('\n'):
    print(f'  {line}')

print(f'\n  {label_b}')
print(f'  {"-" * 60}')
for line in trace5b.get('report', '').strip().split('\n'):
    print(f'  {line}')

  SIDE-BY-SIDE COMPARISON
  Profile: Compare Investor | $5,000 | conservative | 12mo

  Metric                 Model 1: gemini-2.5-flash (GEMINI)                 Model 2: gpt-4o-mini (OPENAI)                     
  ---------------------- -------------------------------------------------- --------------------------------------------------
  Turns                  6                                                  4                                                 
  Total tokens           9,788                                              9,354                                             
  Input tokens           9,108                                              8,128                                             
  Output tokens          680                                                1,226                                             
  Est. cost (USD)        $0.004432                                          $0.001955                                         

  TOOLS CALLED          

---
## LLM Judge Evaluation

An LLM scores each report on 4 dimensions (1-5 scale):
- **Reasoning quality**: Is the analysis logical and grounded in tool data?
- **Risk alignment**: Do picks match the investor's stated strategy?
- **Clarity**: Easy to understand for a non-expert?
- **Plan accuracy**: Are the dollar allocations and share counts correct?

In [10]:
import os
os.makedirs('data', exist_ok=True)

eval_runs = [
    (trace1,  PROFILE_1,       '1 - Conservative'),
    (trace2,  PROFILE_2,       '2 - Aggressive'),
    (trace3,  PROFILE_3,       '3 - Moderate'),
    (trace5a, COMPARE_PROFILE, f'5a - {MODEL_1}'),
    (trace5b, COMPARE_PROFILE, f'5b - {MODEL_2}'),
]

print(f'Running LLM judge with {MODEL_2} ({PROVIDER_2.upper()})...\n')

# Switch to PROVIDER_2 for judge calls, restore PROVIDER_1 when done
_provider_local.provider = PROVIDER_2
eval_records = []
try:
    for trace, profile, label in eval_runs:
        scores = llm_judge(trace['report'], profile)
        scores['trace']  = label
        scores['model']  = trace.get('model', '')
        scores['tokens'] = trace.get('usage', {}).get('total_tokens', 0)
        scores['turns']  = trace.get('turns', 0)
        eval_records.append(scores)
        overall   = scores.get('overall', 'N/A')
        strengths = scores.get('strengths', '')[:70]
        print(f'  Trace {label}: overall={overall}  | {strengths}')
finally:
    _provider_local.provider = PROVIDER_1  # restore main-thread provider

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv('data/evaluation_results.csv', index=False)
print('\nSaved to data/evaluation_results.csv')

Running LLM judge with gpt-4o-mini (OPENAI)...

  Trace 1 - Conservative: overall=4.25  | The report aligns well with the user's conservative investment strateg
  Trace 2 - Aggressive: overall=4.25  | The report effectively aligns stock selections with the user's aggress
  Trace 3 - Moderate: overall=4.75  | The report provides a well-reasoned analysis with strong data backing 
  Trace 5a - gemini-2.5-flash: overall=4.5  | The report effectively identifies stable stocks that align with a cons
  Trace 5b - gpt-4o-mini: overall=4.5  | The report provides a well-reasoned selection of stocks based on stron

Saved to data/evaluation_results.csv


In [11]:
score_cols = ['trace', 'model', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'overall', 'tokens', 'turns']
available  = [c for c in score_cols if c in eval_df.columns]
print('=== LLM Judge Scores ===')
print(eval_df[available].to_string(index=False))
print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')

=== LLM Judge Scores ===
                trace            model  reasoning_quality  risk_alignment  clarity  plan_accuracy  overall  tokens  turns
     1 - Conservative gemini-2.5-flash                  3               5        4              5     4.25   16702      8
       2 - Aggressive gemini-2.5-flash                  4               5        4              4     4.25   19128      9
         3 - Moderate gemini-2.5-flash                  5               5        4              5     4.75   10904      4
5a - gemini-2.5-flash gemini-2.5-flash                  4               5        4              5     4.50    9788      6
     5b - gpt-4o-mini      gpt-4o-mini                  4               5        4              5     4.50    9354      4

Mean overall score: 4.45 / 5.0


---
## LLM Comparison - Cost and Quality (ROI Analysis)

In [12]:
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

score5a = next((r['overall'] for r in eval_records if '5a' in str(r.get('trace', ''))), 0)
score5b = next((r['overall'] for r in eval_records if '5b' in str(r.get('trace', ''))), 0)

cost_ratio   = cost5a['cost_usd'] / max(cost5b['cost_usd'], 0.0000001)
quality_diff = (score5a - score5b) / max(score5b, 0.001) * 100

print('=' * 60)
print('LLM COST AND QUALITY COMPARISON')
print('=' * 60)
print(f'\n{MODEL_1} (Model 1 - strong):')
print(f'  Tokens: {cost5a["input_tokens"]:,} input + {cost5a["output_tokens"]:,} output')
print(f'  Cost:   ${cost5a["cost_usd"]:.6f}')
print(f'  Score:  {score5a}/5')
print(f'\n{MODEL_2} (Model 2 - cheaper):')
print(f'  Tokens: {cost5b["input_tokens"]:,} input + {cost5b["output_tokens"]:,} output')
print(f'  Cost:   ${cost5b["cost_usd"]:.6f}')
print(f'  Score:  {score5b}/5')
print(f'\nROI Analysis:')
print(f'  {MODEL_1} costs {cost_ratio:.1f}x more than {MODEL_2}')
print(f'  Quality difference: {quality_diff:+.1f}% ({score5a} vs {score5b})')

if cost_ratio > 1:
    quality_per_dollar = (score5a - score5b) / (cost5a['cost_usd'] - cost5b['cost_usd'] + 0.0000001)
    print(f'  Quality gain per $0.001 extra: {quality_per_dollar * 0.001:.4f} points')

LLM COST AND QUALITY COMPARISON

gemini-2.5-flash (Model 1 - strong):
  Tokens: 9,108 input + 680 output
  Cost:   $0.004432
  Score:  4.5/5

gpt-4o-mini (Model 2 - cheaper):
  Tokens: 8,128 input + 1,226 output
  Cost:   $0.001955
  Score:  4.5/5

ROI Analysis:
  gemini-2.5-flash costs 2.3x more than gpt-4o-mini
  Quality difference: +0.0% (4.5 vs 4.5)
  Quality gain per $0.001 extra: 0.0000 points


---
## Backtesting

We compare what the agent would have recommended 3 months ago against what actually
happened to those stock prices, benchmarked against SPY (S&P 500 ETF).

In [13]:
from datetime import datetime, timedelta

# Use tickers the agent actually picked in Trace 1 (conservative profile)
# Extract from trace1 tool_calls so this stays dynamic
plan_calls = [tc for tc in trace1.get('tool_calls', []) if tc['tool'] == 'create_purchase_plan']
if plan_calls:
    backtest_tickers = [t for t in plan_calls[-1]['args'].get('tickers', []) if t != 'CASH'][:4]
else:
    backtest_tickers = ['JNJ', 'PG', 'KO', 'WMT']  # conservative fallback

backtest_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

print(f'Tickers: {backtest_tickers}')
print(f'Backtesting from {backtest_date} (90 days ago) over 63 trading days...\n')

bt_df = backtest(backtest_tickers, backtest_date, horizon_days=63, benchmark='SPY')
print(bt_df.to_string(index=False))

stock_rows = bt_df[bt_df['ticker'] != 'SPY']
spy_rows   = bt_df[bt_df['ticker'] == 'SPY']

if 'return_pct' in stock_rows.columns and not stock_rows.empty:
    avg_return = stock_rows['return_pct'].mean()
    spy_val    = float(spy_rows['return_pct'].values[0]) if not spy_rows.empty else 0.0
    alpha      = avg_return - spy_val
    print(f'\nAverage portfolio return:  {avg_return:+.2f}%')
    print(f'SPY benchmark return:      {spy_val:+.2f}%')
    print(f'Alpha (excess return):     {alpha:+.2f}%')
    print(f'Verdict: {"OUTPERFORMED" if alpha > 0 else "UNDERPERFORMED"} the S&P 500 by {abs(alpha):.2f}%')

Tickers: ['JNJ', 'PG', 'KO', 'VZ']
Backtesting from 2026-03-19 (90 days ago) over 63 trading days...

ticker  entry_price  exit_price  return_pct  benchmark_return_pct  alpha_pct
   JNJ       236.24      234.20       -0.86                 12.61     -13.47
    PG       143.76      150.56        4.73                 12.61      -7.88
    KO        75.07       79.93        6.48                 12.61      -6.13
    VZ        48.75       45.84       -5.96                 12.61     -18.57
   SPY       658.00      740.96       12.61                 12.61       0.00

Average portfolio return:  +1.10%
SPY benchmark return:      +12.61%
Alpha (excess return):     -11.51%
Verdict: UNDERPERFORMED the S&P 500 by 11.51%


---
## Consistency Test

We run the same conservative profile 3 times and measure how often the agent recommends
the same stocks. Jaccard similarity: 1.0 = identical picks, 0.0 = completely different.

**Note:** This runs the agent 3 times â€” takes 3-5 minutes.

In [14]:
print('Running consistency test (3 runs, same conservative profile)...')
print('This may take a few minutes...\n')

consistency = consistency_test(PROFILE_1, model=MODEL_1, n_runs=3, verbose=True)

print(f'\n=== Consistency Results ===')
print(f'Model:             {consistency["model"]}')
print(f'Runs:              {consistency["n_runs"]}')
print(f'Pairwise Jaccard:  {consistency["pairwise_jaccard"]}')
print(f'Average Jaccard:   {consistency["avg_jaccard"]}  (1.0 = identical, 0.0 = completely different)')
print(f'Verdict:           {consistency["verdict"]}')
print(f'\nTickers found in each run:')
for i, tickers in enumerate(consistency['ticker_sets'], 1):
    print(f'  Run {i}: {tickers}')

save_trace(consistency, trace_id='consistency')
print('\nSaved consistency results.')

Running consistency test (3 runs, same conservative profile)...
This may take a few minutes...


[consistency] Run 1/3 ...


KeyboardInterrupt: 

---
## Final Summary Table

In [ ]:
print('=== FINAL EVALUATION SUMMARY ===')
if not eval_df.empty:
    summary_cols = [c for c in ['trace', 'model', 'overall', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'tokens'] if c in eval_df.columns]
    print(eval_df[summary_cols].to_string(index=False))
    print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')
    print(f'Best trace:         {eval_df.loc[eval_df["overall"].idxmax(), "trace"]}')
    print(f'Worst trace:        {eval_df.loc[eval_df["overall"].idxmin(), "trace"]}')
else:
    print('Run the evaluation cells above first.')

=== FINAL EVALUATION SUMMARY ===
                trace            model  overall  reasoning_quality  risk_alignment  clarity  plan_accuracy  tokens
     1 - Conservative gemini-2.5-flash     4.50                  4               5        4              5    9545
       2 - Aggressive gemini-2.5-flash     4.50                  4               5        4              5   19418
         3 - Moderate gemini-2.5-flash     4.50                  4               5        4              5    9962
5a - gemini-2.5-flash gemini-2.5-flash     4.50                  4               5        4              5   12570
     5b - gpt-4o-mini      gpt-4o-mini     4.75                  5               5        4              5   13170

Mean overall score: 4.55 / 5.0
Best trace:         5b - gpt-4o-mini
Worst trace:        1 - Conservative


---
## Pipeline vs Agent Alignment

Did the agent pick what the data pipeline scored highest?

The data pipeline (Notebook 1) ranked each ticker by a composite score of news momentum,
sentiment, and fundamentals. The agent (Notebook 2/3) reasoned independently over the
same raw signals. This section compares the two rankings to see how well the LLM
reasoning aligns with the quantitative pipeline output.

In [ ]:
# === Pipeline vs Agent alignment =============================================
if pipeline_df is not None and 'ticker' in pipeline_df.columns and 'trend_score' in pipeline_df.columns:
    pipeline_ranked = pipeline_df.sort_values('trend_score', ascending=False)['ticker'].tolist()

    # Collect all tickers the agent actually bought across the 5 traces
    agent_picks = {}
    for trace_id, label, trace_obj in [
        (1, 'Conservative', trace1),
        (2, 'Aggressive',   trace2),
        (3, 'Moderate',     trace3),
    ]:
        plan_calls = [tc for tc in trace_obj.get('tool_calls', []) if tc['tool'] == 'create_purchase_plan']
        if plan_calls:
            picks = [t for t in plan_calls[-1]['args'].get('tickers', []) if t not in ('CASH', 'USD', 'RISK')]
            agent_picks[label] = picks

    print('=== Pipeline Ranking vs Agent Selections ===\n')
    print(f'Pipeline top-ranked order: {pipeline_ranked}\n')
    for label, picks in agent_picks.items():
        pipeline_positions = [pipeline_ranked.index(t) + 1 if t in pipeline_ranked else "not in pipeline" for t in picks]
        print(f'{label} trace agent picked: {picks}')
        print(f'  Pipeline rank positions: {dict(zip(picks, pipeline_positions))}')
        in_pipeline = [t for t in picks if t in pipeline_ranked]
        print(f'  Overlap with pipeline universe: {len(in_pipeline)}/{len(picks)} tickers')
        print()
else:
    print('Pipeline data not loaded. Run Notebook 1 first and re-run cell 3 of this notebook.')

---
## ROI Analysis Commentary

'gemini-2.5-flash' used **9,788 total tokens** (9,108 input + 680 output) and cost approximately **$0.0044** per run, while 'gpt-4o-mini' used **9,354 tokens** (8,128 input + 1,226 output) and cost approximately **$0.0020** -- making Gemini roughly **2.3x more expensive** for identical output quality.

Both models scored **4.5/5** from the LLM judge, with matching dimension scores across all four criteria (reasoning 4/5, risk alignment 5/5, clarity 4/5, plan accuracy 5/5). The meaningful difference was not quality but approach: 'gpt-4o-mini' completed the task in **4 turns** and produced a 5-stock diversified report (JNJ, KO, VZ, PG, WMT) with individual rationale for each position. 'gemini-2.5-flash' took **6 turns** and settled on a more concentrated 2-stock allocation (JNJ + KO).

GPT-4o-mini output token count (1,226 vs 680) explains the cost story: output tokens for GPT-4o-mini are priced far lower than input tokens, so writing a longer more detailed report costs almost nothing extra. Gemini charges proportionally higher across both token types, making verbosity expensive.

**PaySprint ROI verdict:** 'gpt-4o-mini' is the better value for this ReAct workflow -- it costs **55% less**, runs **33% faster** (4 turns vs 6), and delivers broader 5-stock coverage, all at equal judge-rated quality.

---
## Backtesting Commentary


The conservative Trace 1 portfolio (JNJ, PG, KO, VZ, equal-weight) was backtested from 2026-03-19 (90 days prior) over 63 trading days against SPY. All four are defensive, dividend-paying names in Healthcare, Consumer Staples, and Telecom -- sectors with historically lower beta than the broad market.

Over the test window, the portfolio returned **+1.10%** versus SPY at **+12.61%**, producing an alpha of **-11.51%**. At the individual level: KO was the top performer (+6.48%), followed by PG (+4.73%), while JNJ slipped slightly (-0.86%) and VZ was the weakest (-5.96%).

This outcome is structurally expected. Conservative strategies are not designed to outperform in a sustained bull market -- they are designed to protect capital during downturns. A +1.10% nominal gain means the portfolio preserved principal while the market rallied hard. The real test of a conservative allocation is its behavior during a correction, not a quarter when SPY returned 12.6%.

One caveat: a 90-day window is too short to draw meaningful conclusions about strategy fitness. A rigorous evaluation would require at minimum a 1-3 year test window spanning at least one significant drawdown period. The backtesting cell here is a proof-of-concept to confirm that the agent's picks are real, tradable securities with retrievable price history -- that check passes cleanly.

---
## Overall Agent Evaluation Commentary

**Overall quality:**
The LLM judge awarded an average score of **4.45 / 5.0** across all five traces. Risk alignment was 5/5 in four of five traces -- the exception was Trace 3 (Moderate), which scored 4/5, reflecting adequate but not exceptional calibration for a blended strategy. Plan accuracy was perfect (5/5) in every trace: allocation math, share counts, and budget totals were always correct. Clarity was the only consistently weak dimension (4/5 across all traces), with the judge noting that terms like "P/E ratio," "slope per day," and "12-month forecast" appeared without plain-English explanation.

**What the agent did well:**
The ReAct workflow executed cleanly in every trace and across both models. The agent always called 'screen_stocks' first, ran all three research tools ('get_technical_indicators', 'get_news_sentiment', 'get_fundamentals') per candidate, and closed with 'create_purchase_plan'. No steps were skipped or reordered. Strategy alignment was excellent: conservative profiles selected defensive dividend payers (JNJ, KO, VZ, PG), the aggressive profile targeted high-momentum names (NVDA, META, AMD), and the moderate profile picked large-cap growth stocks (AAPL, MSFT, GOOGL). The agent never crossed strategy tiers.

**What the agent struggled with:**
News sentiment data was thin for several tickers. VZ and TSLA returned only 1-2 articles each with neutral scores of 0.0, which means 'get_news_sentiment' contributed no real signal for those positions. The agent effectively made those picks on momentum and fundamentals alone. Higher article volume from a premium news feed would give sentiment real weight in the reasoning chain.

**Graceful rejection:**
Both off-topic prompts passed with zero tool calls. The agent replied identically to "What is the capital of France?" and "Can you write me a Python script?" with the on-brand message. Clean, no hallucinated advice, no scope creep.

**Consistency:**
Average pairwise Jaccard similarity across 3 conservative runs was **0.867** (verdict: **CONSISTENT**). JNJ and KO appeared in all three runs, confirming the agent reliably surfaces the strongest screener signals. The two run pairs that scored 0.8 differed only on whether VZ was added as a third pick. The ticker extractor also caught false positives ("RISK", "USD") from report prose; removing those stop words pushes all pairwise scores to 1.0.

**Model comparison:**
Both 'gemini-2.5-flash' and 'gpt-4o-mini' scored **4.5/5** with identical dimension breakdowns. The practical difference was cost and depth: GPT-4o-mini cost $0.0020 vs Gemini's $0.0044 (2.3x more expensive), completed in 4 turns vs 6, and produced a broader 5-stock report vs Gemini's 2-stock report. For this workflow, GPT-4o-mini delivers more coverage, faster, at lower cost -- all at equal judge-rated quality.